# Demo 3: Self-Consistency & Inference Scaling Law

本 notebook 包含：
1. Self-Consistency 多数投票效果模拟
2. Inference Scaling Law 曲线 → `assets/chart-scaling-law.png`

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import numpy as np

# 注册中文字体
fm.fontManager.addfont('/home/zhangboju/.fonts/NotoSansSC.ttf')

matplotlib.rcParams.update({
    'figure.facecolor': '#1a2332',
    'axes.facecolor': '#1a2332',
    'axes.edgecolor': '#8faab0',
    'axes.labelcolor': '#f1faee',
    'text.color': '#f1faee',
    'xtick.color': '#8faab0',
    'ytick.color': '#8faab0',
    'grid.color': '#223044',
    'legend.facecolor': '#223044',
    'legend.edgecolor': '#8faab0',
    'font.size': 12,
    'font.family': 'Noto Sans SC',
    'axes.unicode_minus': False,
})

## 1. Self-Consistency 投票效果

In [ ]:
def majority_vote_accuracy(p_single, N):
    """
    模拟多数投票的准确率
    p_single: 单次推理准确率
    N: 采样次数
    """
    from scipy.stats import binom
    accuracy = 0
    for k in range(N // 2 + 1, N + 1):
        accuracy += binom.pmf(k, N, p_single)
    return accuracy

# 模拟
p_single = 0.57  # LLaMA-2-70B 在 GSM8K 上的 8-shot CoT 准确率 (Touvron et al. 2023)
N_values = list(range(1, 51))
accuracies = [majority_vote_accuracy(p_single, N) for N in N_values]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(N_values, accuracies, color='#5ec4c4', linewidth=2.5)
ax.fill_between(N_values, accuracies, alpha=0.1, color='#5ec4c4')
ax.axhline(y=p_single, color='#e09060', linestyle='--', alpha=0.6, label=f'单次推理: {p_single:.0%}')

# 标注关键点（使用奇数 N 避免偶数 N 的多数阈值问题）
for N_mark in [1, 5, 11, 21, 41]:
    idx = N_mark - 1
    ax.plot(N_mark, accuracies[idx], 'o', color='#a8dadc', markersize=7, zorder=5)
    ax.annotate(f'N={N_mark}: {accuracies[idx]:.1%}',
                (N_mark, accuracies[idx]),
                textcoords='offset points', xytext=(5, 10),
                fontsize=9, color='#a8dadc')

ax.set_xlabel('采样次数 N')
ax.set_ylabel('准确率')
ax.set_title('Self-Consistency: 多数投票准确率 vs 采样次数', fontsize=13, color='#a8dadc', pad=10)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.3, 1.0)

plt.tight_layout()
plt.savefig('../assets/chart-self-consistency-voting.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart-self-consistency-voting.png')

## 2. Inference Scaling Law 曲线

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

# 模拟 inference compute (FLOPs) vs accuracy for different model sizes
# 使用对数线性关系: accuracy = a * log(compute) + b
compute = np.logspace(12, 16, 100)  # FLOPs

models_config = {
    '1B model': {'a': 6.5, 'b': -70, 'color': '#e09060'},
    '7B model': {'a': 7.5, 'b': -75, 'color': '#2d8b8b'},
    '70B model': {'a': 8.5, 'b': -80, 'color': '#5ec4c4'},
    '405B model': {'a': 9.0, 'b': -82, 'color': '#a8dadc'},
}

for name, cfg in models_config.items():
    accuracy = cfg['a'] * np.log10(compute) + cfg['b']
    accuracy = np.clip(accuracy, 0, 100)
    ax.semilogx(compute, accuracy, color=cfg['color'], linewidth=2.5, label=name)

# Annotate key insight
ax.annotate('小模型 + 更多推理计算\n≈ 大模型性能',
             xy=(1e15, 72), xytext=(5e13, 55),
             fontsize=11, color='#e0b060', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#e0b060', lw=1.5))

# Mark single-inference points
single_points = {
    '1B': (1e13, 10),
    '7B': (1e14, 20),
    '70B': (1e15, 30),
}
for name, (x, y) in single_points.items():
    ax.plot(x, x/1e14 + 10, 's', color='#f1faee', markersize=5, alpha=0.5)

ax.set_xlabel('推理计算量 (FLOPs)', fontsize=12)
ax.set_ylabel('性能 (Accuracy %)', fontsize=12)
ax.set_title('Inference-Time Scaling Law', fontsize=14, color='#a8dadc', pad=10)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('../assets/chart-scaling-law.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart-scaling-law.png')